In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from skopt.space import Real, Integer
from modules.interface import RunModel, RunOptimization
from modules.models import XGBoostTraining
from sklearn.metrics import mean_squared_error
from pytorch_tabnet.tab_model import TabNetRegressor

In [3]:
# --- INÍCIO DO SCRIPT DE TESTE ---

# 1. Carregar o Banco de Dados (California Housing)
print("1. Carregando o banco de dados...")
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target)

# 2. Dividir os dados em Conjunto de Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- OTIMIZAÇÃO PARA LIGHTGBM ---
print("="*50)
print("INICIANDO OTIMIZAÇÃO PARA LIGHTGBM")
print("="*50)

# Instanciando o modelo
clf_tabnet = TabNetRegressor(verbose=0)

# IMPORTANTE: TabNet exige arrays do NumPy (não DataFrames).
# Além disso, o target (y) precisa ser um array 2D (.reshape(-1, 1)).
clf_tabnet.fit(
    X_train=X_train.values, 
    y_train=y_train.values.reshape(-1, 1),
    eval_set=[(X_test.values, y_test.values.reshape(-1, 1))]
)

preds_tabnet = clf_tabnet.predict(X_test.values)
rmse_tabnet = np.sqrt(mean_squared_error(y_test, preds_tabnet))
print(f"\n-> RMSE do TabNet: {rmse_tabnet:.4f}")

1. Carregando o banco de dados...
INICIANDO OTIMIZAÇÃO PARA LIGHTGBM


/media/ricardo/HD/Doutorado/calibration/calibration_refactor/calibration_refactor_main/venv/lib/python3.13/site-packages/torch/utils/data/_utils/collate.py:288: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return collate([torch.as_tensor(b) for b in batch], collate_fn_map=collate_fn_map)



Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_0_mse = 8.77501


/media/ricardo/HD/Doutorado/calibration/calibration_refactor/calibration_refactor_main/venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



-> RMSE do TabNet: 2.9623


In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Carregar o Banco de Dados (California Housing)
print("1. Carregando o banco de dados...")
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target)

# 2. Dividir os dados em Conjunto de Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =======================================================
# OPÇÃO 1: SE VOCÊ REALMENTE QUERIA USAR O LIGHTGBM
# =======================================================
print("\n" + "="*50)
print("INICIANDO OTIMIZAÇÃO PARA LIGHTGBM")
print("="*50)

# Import correto para o LightGBM
from lightgbm import LGBMRegressor

# Instanciando o modelo
clf_lgb = LGBMRegressor(random_state=42)

# O LightGBM aceita DataFrames do Pandas nativamente
clf_lgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)]
)

preds_lgb = clf_lgb.predict(X_test)
rmse_lgb = np.sqrt(mean_squared_error(y_test, preds_lgb))
print(f"\n-> RMSE do LightGBM: {rmse_lgb:.4f}")


# =======================================================
# OPÇÃO 2: SE VOCÊ REALMENTE QUERIA USAR O TABNET
# =======================================================
print("\n" + "="*50)
print("INICIANDO OTIMIZAÇÃO PARA TABNET")
print("="*50)

# Import correto para o TabNet
from pytorch_tabnet.tab_model import TabNetRegressor

# Instanciando o modelo
clf_tabnet = TabNetRegressor(verbose=0)

# IMPORTANTE: TabNet exige arrays do NumPy (não DataFrames).
# Além disso, o target (y) precisa ser um array 2D (.reshape(-1, 1)).
clf_tabnet.fit(
    X_train=X_train.values, 
    y_train=y_train.values.reshape(-1, 1),
    eval_set=[(X_test.values, y_test.values.reshape(-1, 1))]
)

preds_tabnet = clf_tabnet.predict(X_test.values)
rmse_tabnet = np.sqrt(mean_squared_error(y_test, preds_tabnet))
print(f"\n-> RMSE do TabNet: {rmse_tabnet:.4f}")

1. Carregando o banco de dados...

INICIANDO OTIMIZAÇÃO PARA LIGHTGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002149 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.071947

-> RMSE do LightGBM: 0.4635

INICIANDO OTIMIZAÇÃO PARA TABNET

Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_0_mse = 8.77501


/media/ricardo/HD/Doutorado/calibration/calibration_refactor/calibration_refactor_main/venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



-> RMSE do TabNet: 2.9623
